# 🔬 LoComo Benchmark: Memlayer vs Mem0 (Real Dataset)

This notebook runs the **official LoComo benchmark** comparing:

1. **Mem0** - Baseline vector-only system
2. **Memlayer (Optimized)** - Custom salience config (threshold=0.1)

## 📊 Dataset

Using **real LoComo data** from ACL 2024 paper:
- Source: https://github.com/snap-research/locomo
- Handles both string and numeric answers (e.g., years like `2022`)

**⏱️ Estimated Runtime**: 15-20 minutes (5 conversations)

## 🛠️ Setup & Installation

In [ ]:
# Install dependencies
print("📦 Installing dependencies...\n")
!pip install -q git+https://github.com/thebnbrkr/memlayer.git
!pip install -q mem0ai rouge-score pandas matplotlib seaborn
print("✅ Installation complete!")

In [ ]:
import os
from getpass import getpass
if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass('🔑 Enter OpenAI API key: ')
print("✅ API key configured!")

## 📥 Load Real LoComo Dataset

In [ ]:
import json, requests

def download_locomo_dataset(dataset_name="locomo10.json"):
    url = f"https://raw.githubusercontent.com/snap-research/locomo/refs/heads/main/data/{dataset_name}"
    print(f"📥 Downloading {dataset_name}...\n")
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        print(f"✅ Downloaded {len(data)} conversations")
        return data
    except Exception as e:
        print(f"❌ Error: {e}\n⚠️  Using fallback data")
        return None

def convert_locomo_format(locomo_data):
    converted = []
    for idx, item in enumerate(locomo_data):
        conversation = [turn.get("text", "") for turn in item.get("dialogue", [])]
        qa_pairs = [{
            "question": qa.get("question", ""),
            "ground_truth_answer": qa.get("answer", ""),
            "evidence": qa.get("evidence", []),
            "category": qa.get("category", 0)
        } for qa in item.get("qa", [])]
        converted.append({
            "conversation_id": f"conv_{idx:03d}",
            "conversation": conversation,
            "qa_pairs": qa_pairs
        })
    return converted

print("="*70)
print("📊 LOADING LOCOMO DATASET")
print("="*70 + "\n")

locomo_raw = download_locomo_dataset("locomo10.json")
if locomo_raw:
    conversations = convert_locomo_format(locomo_raw)[:5]  # First 5
    print(f"\n✅ Prepared {len(conversations)} conversations")
else:
    conversations = [{"conversation_id": "conv_001", "conversation": ["Sample"], "qa_pairs": [{"question": "Test?", "ground_truth_answer": "Sample", "category": 1}]}]

total_qa = sum(len(c['qa_pairs']) for c in conversations)
print(f"\n📈 Stats: {len(conversations)} conversations, {total_qa} QA pairs")
print("\n" + "="*70)

## 📏 Evaluation Metrics (Fixed for Numeric Answers)

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

def calculate_f1(prediction, ground_truth) -> float:
    """Calculate F1 score. Handles strings AND numbers (e.g., year 2022)."""
    # Convert to string if needed (fixes integer answers!)
    if not isinstance(prediction, str):
        prediction = str(prediction)
    if not isinstance(ground_truth, str):
        ground_truth = str(ground_truth)
    
    pred_tokens = set(prediction.lower().split())
    truth_tokens = set(ground_truth.lower().split())
    
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 0.0
    
    common = pred_tokens & truth_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * (precision * recall) / (precision + recall)

def calculate_rouge(prediction, ground_truth) -> dict:
    """Calculate ROUGE scores. Handles strings AND numbers."""
    # Convert to string if needed
    if not isinstance(prediction, str):
        prediction = str(prediction)
    if not isinstance(ground_truth, str):
        ground_truth = str(ground_truth)
    
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(ground_truth, prediction)
    
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }

print("✅ Metrics defined (handles both string & numeric answers!)")

## 🧪 Test 1: Mem0 (Baseline)

In [ ]:
from mem0 import Memory
import time

mem0_config = {"vector_store": {"provider": "chroma", "config": {"collection_name": "mem0_locomo", "path": "./mem0_storage"}}}
mem0_client = Memory.from_config(mem0_config)

print("="*70)
print("🧪 TEST 1: MEM0")
print("="*70 + "\n")

mem0_results = []
for conv_idx, conversation in enumerate(conversations):
    user_id = f"user_{conv_idx}"
    print(f"\nConversation {conv_idx+1}/{len(conversations)}")
    
    for msg in conversation['conversation']:
        mem0_client.add(msg, user_id=user_id)
        time.sleep(0.1)
    
    for qa in conversation['qa_pairs']:
        results = mem0_client.search(qa['question'], user_id=user_id, limit=5)
        
        if results:
            try:
                if isinstance(results, dict) and 'results' in results:
                    response = " ".join([m.get("memory", str(m)) for m in results['results'][:3]])
                elif isinstance(results, list):
                    response = " ".join([r.get("memory", str(r)) if isinstance(r, dict) else str(r) for r in results[:3]])
                else:
                    response = str(results)
            except:
                response = "Error"
        else:
            response = "No info"
        
        f1 = calculate_f1(response, qa['ground_truth_answer'])
        rouge = calculate_rouge(response, qa['ground_truth_answer'])
        mem0_results.append({'f1': f1, 'rouge1': rouge['rouge1'], 'rouge2': rouge['rouge2'], 'rougeL': rouge['rougeL']})
        print(f"  Q: {qa['question'][:50]}... F1: {f1:.3f}")
        time.sleep(0.5)

mem0_avg_f1 = np.mean([r['f1'] for r in mem0_results])
print(f"\n{'='*70}\n📊 MEM0 AVG F1: {mem0_avg_f1:.3f}\n{'='*70}")

## 🧪 Test 2: Memlayer (Optimized)

In [ ]:
from memlayer import OpenAI as Memlayer
from memlayer.config.salience import TenantSalienceConfig, SalienceComponent, ScoringFunctionType, AdaptiveThresholdConfig, ThresholdStrategy

print("="*70)
print("🧪 TEST 2: MEMLAYER")
print("="*70 + "\n")

memlayer_client = Memlayer(
    model="gpt-4o-mini",
    user_id="bench",
    storage_path="./memlayer_storage",
    operation_mode="online",
    salience_config=TenantSalienceConfig(
        components=[
            SalienceComponent(ScoringFunctionType.KEYWORD_MATCH, 0.5, {"keywords": ["I", "my", "me", "work", "job", "career"], "case_sensitive": False}),
            SalienceComponent(ScoringFunctionType.LENGTH_BONUS, 0.5, {"optimal_length": 50, "steepness": 0.02})
        ],
        threshold_config=AdaptiveThresholdConfig(ThresholdStrategy.ABSOLUTE, absolute_threshold=0.1)
    )
)

memlayer_results = []
for conv_idx, conversation in enumerate(conversations):
    print(f"\nConversation {conv_idx+1}/{len(conversations)}")
    
    for msg in conversation['conversation']:
        memlayer_client.chat([{"role": "user", "content": msg}])
        time.sleep(0.1)
    
    for qa in conversation['qa_pairs']:
        response = memlayer_client.chat([{"role": "user", "content": qa['question']}])
        f1 = calculate_f1(response, qa['ground_truth_answer'])
        rouge = calculate_rouge(response, qa['ground_truth_answer'])
        memlayer_results.append({'f1': f1, 'rouge1': rouge['rouge1'], 'rouge2': rouge['rouge2'], 'rougeL': rouge['rougeL']})
        print(f"  Q: {qa['question'][:50]}... F1: {f1:.3f}")
        time.sleep(0.5)

memlayer_avg_f1 = np.mean([r['f1'] for r in memlayer_results])
print(f"\n{'='*70}\n📊 MEMLAYER AVG F1: {memlayer_avg_f1:.3f}\n{'='*70}")

## 📊 Final Comparison

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame([{'System': 'Mem0', 'F1': mem0_avg_f1}, {'System': 'Memlayer', 'F1': memlayer_avg_f1}])
improvement = ((memlayer_avg_f1 - mem0_avg_f1) / mem0_avg_f1) * 100

print("\n" + "="*70)
print("🏆 FINAL RESULTS")
print("="*70)
print(df.to_string(index=False))
print(f"\n🎯 Improvement: {improvement:+.1f}%")

if improvement > 0:
    print(f"\n✅ SUCCESS! Memlayer BEATS Mem0 by {improvement:.1f}%! 🎉")
else:
    print(f"\n⚠️  Memlayer: {improvement:.1f}% vs Mem0")

df.plot(x='System', kind='bar', y='F1', legend=False)
plt.title('LoComo Benchmark: Memlayer vs Mem0')
plt.ylabel('F1 Score')
plt.ylim(0, 1)
plt.show()
print("="*70)